In [4]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
from qiskit_aer import AerSimulator
import math

In [5]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.
# This notebook is for a simulation of the protocol without an attacker.

**Create Simulator**

In [6]:
simulator = AerSimulator()

**Generate Bitstrings for Alice and Bob**

In [7]:
def generate_bits_quantum(n_bits): # HELPER FUNCTION TO GENERATE THE BITS so it can be any length..

  qc = QuantumCircuit(1, 1)
  qc.h(0)
  qc.measure(0, 0)

  result = simulator.run(qc, shots=n_bits, memory=True).result() # run n_bits times with shots
  bitstring = ''.join(result.get_memory()) # combine together

  return bitstring

alice_basis=generate_bits_quantum(16)
alice_state=generate_bits_quantum(16)
bob_basis=generate_bits_quantum(16)

print("Randomly generating Alice and Bob...\n")
print(f"alice_basis: {alice_basis}")
print(f"alice_state: {alice_state}")
print(f"bob_basis: {bob_basis}")

Randomly generating Alice and Bob...

alice_basis: 0001010101010001
alice_state: 1010101110111100
bob_basis: 1011110000111101


**Function for Alice and Bob to Send and Receive**

In [8]:
"""
LEGEND for BASIS
0 -> standard basis
1 -> diagonal basis

"""

# For Alice
def sender_function(state, basis):
  bitlength = len(state)

  circuit = QuantumCircuit(bitlength)

  for i in range(bitlength):
    # qubit default is ket 0
    if state[i] == '1': # if bit 1, use pauli-x gate to change to ket 1
      circuit.x(i)

    # after that, check the basis of the bit
    if basis[i] == '1': # if bit 1, use hadamard gate to change into ket - or +
      circuit.h(i)

  return circuit

# For Bob
def receiver_function(message, measurement_basis):
  bitlength = len(measurement_basis)

  for i in range(bitlength):

    if measurement_basis[i] == '1':
      message.h(i)

  message.measure_all()

  # Run the measured circuit
  compiled_circuit = transpile(message, simulator)

  result = simulator.run(
        compiled_circuit,
        shots=1,
        memory=True
    ).result()

  return result.get_memory()[0][::-1]

# Simulate Send and Receive
alice_encoded_circuit = sender_function(alice_state, alice_basis)
bob_received_circuit = receiver_function(alice_encoded_circuit, bob_basis)


**Sifting Function** : Generates both sender and receiver key

In [9]:
def sifting_function(sender_basis, sender_state, receiver_basis, receiver_result):
  sender_key = ""
  receiver_key = ""

  for i in range(len(sender_basis)):
    if sender_basis[i] == receiver_basis[i]:
      sender_key += sender_state[i]
      receiver_key += receiver_result[i]

  return sender_key, receiver_key

sifted_key, receiver_key = sifting_function(alice_basis, alice_state, bob_basis, bob_received_circuit)

**Check Quantum Bit Error Rate**

In [10]:
def calculate_qber(sender_key, receiver_key):
  errors = 0

  for i in range(len(sender_key)):
    if sender_key[i] != receiver_key[i]:
      errors += 1

  if len(sender_key) == 0:
    qber = 0
  else:
    qber = errors / len(sender_key)

  return qber, errors

qber, errors = calculate_qber(sifted_key, receiver_key)
print(f"Quantum Bit Error Rate: {qber}")
print(f"Number of errors: {errors}")

Quantum Bit Error Rate: 0.0
Number of errors: 0


Final Results

In [11]:
# Final Conclusion Block

print("\n==============================")
print("BB84 Protocol Plain Summary")
print("==============================")

print("\n Bitstring and Basis generation with Quantum Randomness")
print("----------------------------------")

print("\nAlice basis: ", alice_basis)
print("Alice state: ", alice_state)
print("Bob basis:   ", bob_basis)

print("Sifted key:  ", sifted_key)
print("Receiver key:", receiver_key)
print("Sifted key length:", len(sifted_key))

print("\n QBER Checking")
print("----------------")
print("Number of errors:", errors)
print("QBER:", qber)
print("QBER percentage:", qber * 100, "%")



BB84 Protocol Plain Summary

 Bitstring and Basis generation with Quantum Randomness
----------------------------------

Alice basis:  0001010101010001
Alice state:  1010101110111100
Bob basis:    1011110000111101
Sifted key:   00011100
Receiver key: 00011100
Sifted key length: 8

 QBER Checking
----------------
Number of errors: 0
QBER: 0.0
QBER percentage: 0.0 %
